In [0]:
def export_table_to_csv(table_name, volume_output_path):
    from pyspark.sql import SparkSession
    from pyspark.sql.functions import col, to_json
    spark = SparkSession.builder.getOrCreate()
    
    # Extrai o nome limpo da tabela para o arquivo final
    # Ex: 'workspace.default.posts_creator' -> 'posts_creator.csv'
    clean_table_name = table_name.split(".")[-1]
    final_filename = f"{clean_table_name}.csv"
    
    # 1. Lê a tabela Delta atualizada pelo Autoloader
    df = spark.table(table_name)
    
    # 2. Trata colunas complexas (Structs/Arrays) para String JSON
    final_cols = [
        to_json(col(f.name)).alias(f.name) if str(f.dataType).startswith(("Struct", "Array")) 
        else col(f.name) for f in df.schema.fields
    ]
    
    # 3. Gravação Temporária (Coalesce 1 para gerar um único arquivo)
    temp_path = f"{volume_output_path}/_temp_{clean_table_name}"
    (df.select(final_cols)
       .coalesce(1)
       .write
       .mode("overwrite")
       .option("header", "true")
       .option("sep", ";")
       .csv(temp_path))
    
    # 4. Localiza o arquivo de dados e move para a pasta final com o nome correto
    files = dbutils.fs.ls(temp_path)
    part_file = [f.path for f in files if f.name.startswith("part-")][0]
    
    dest_path = f"{volume_output_path}/{final_filename}"
    dbutils.fs.cp(part_file, dest_path)
    
    # 5. Limpeza de metadados temporários
    dbutils.fs.rm(temp_path, True)
    
    print(f"✅ Exportação concluída: {final_filename}")
    return dest_path